# Agregations from silver MinIO layer to gold MinIO layer

### Install python dotenv for get the environment variables

In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Imports spark, Dotenv, functions and configurations

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import logging
# Import for get the environment variables 
from dotenv import load_dotenv

import os
import sys
sys.path.append(os.path.abspath("../../")) # Used to reconfigure the absolut path. In this case, setting the absolut path to 2 folders back (notebooks/...) 
from configurations import configurations as config_file # Import configurations.py from the configurations folder
from functions import functions as func_file # Import functions.py from the functions folder

### Import environment variables

In [3]:
load_dotenv()

MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")
MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
POSTGRES_USER=os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD=os.getenv("POSTGRES_PASSWORD")
POSTGRES_CONTAINER=os.getenv("POSTGRES_CONTAINER")

### Configurations from spark

In [4]:
conf = SparkConf()

conf.setAppName("Agregations from MinIO Silver to MinIO Gold") # Spark application name, Usefull for logs
# Add the jars from hadoop-aws and aws-java-sdk-bundle is necessary for org.apache.hadoop.fs.s3a.S3AFileSystem,
# add the Postgresql JDBC jar is necessary for connect on database. Add the delta-spark is necessary for delta catalog, all this Jars is auto-download from spark
conf.set("spark.master", "spark://spark-master:7077") # set the Spark container to be distributed among the workers
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", MINIO_USER) # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", MINIO_PASSWORD) # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True) # Enforces the use of URLs as the format. Without this, Spark attempts to use the AWS standard (bucket.endpoint), which fails in MinIO
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).getOrCreate()


In [5]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

logging.info("Starting Agregations from MinIO Silver to MinIO Gold...")

2026-05-08 22:15:22,675 - INFO - Starting Agregations from MinIO Silver to MinIO Gold...


In [6]:
for table_name in config_file.queries_gold.keys():

    queries_tables = config_file.queries_gold
    silver_path = config_file.data_lakehouse_path['silver']
    gold_path = config_file.data_lakehouse_path['gold']
    output_path = f"{gold_path}gold_{table_name}"
    
    try:
        logging.info(f"processing table {table_name}")

        # Getting query transform
        query = func_file.get_query(table_name, queries_tables, silver_path)

        # Execute query in table from minio bronze
        logging.info(f"Executing query on table {table_name}...")
        dataframe = spark.sql(query)

        # Adding a new column date related the load data on Minio Gold
        dataframe_with_last_update = func_file.add_data_last_update(dataframe)

        # writing the table transformation to the silver layer
        logging.info(f"Writing table {table_name}...")
        dataframe_with_last_update.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)
        logging.info(f"Agregation table {table_name} Completed and saved on: {output_path}")

    
    except Exception as e:
        logging.info(f"Error processing {table_name}: {str(e)}")

logging.info(f"Agreagations to Gold layer Completed")

2026-05-08 22:15:22,684 - INFO - processing table sales_by_country
2026-05-08 22:15:22,685 - INFO - Executing query on table sales_by_country...
2026-05-08 22:18:20,579 - INFO - Writing table sales_by_country...
2026-05-08 22:19:13,050 - INFO - Agregation table sales_by_country Completed and saved on: s3a://gold/adventureworks/gold_sales_by_country
2026-05-08 22:19:13,051 - INFO - processing table sales_per_customer
2026-05-08 22:19:13,052 - INFO - Executing query on table sales_per_customer...
2026-05-08 22:19:13,798 - INFO - Writing table sales_per_customer...
2026-05-08 22:19:22,747 - INFO - Agregation table sales_per_customer Completed and saved on: s3a://gold/adventureworks/gold_sales_per_customer
2026-05-08 22:19:22,748 - INFO - processing table sales_per_employee
2026-05-08 22:19:22,748 - INFO - Executing query on table sales_per_employee...
2026-05-08 22:19:23,354 - INFO - Writing table sales_per_employee...
2026-05-08 22:19:29,874 - INFO - Agregation table sales_per_employee C

In [10]:
df = spark.read.format("delta").load("s3a://gold/adventureworks/gold_sales_per_employee").show(truncate=False)

Py4JJavaError: An error occurred while calling o119.load.
: java.lang.IllegalStateException: No active or default Spark session found
	at org.apache.spark.sql.SparkSession$.$anonfun$active$2(SparkSession.scala:1202)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.SparkSession$.$anonfun$active$1(SparkSession.scala:1202)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.SparkSession$.active(SparkSession.scala:1201)
	at org.apache.spark.sql.delta.sources.DeltaDataSource.getTable(DeltaDataSource.scala:69)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.getTableFromProvider(DataSourceV2Utils.scala:92)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.loadV2Source(DataSourceV2Utils.scala:140)
	at org.apache.spark.sql.DataFrameReader.$anonfun$load$1(DataFrameReader.scala:210)
	at scala.Option.flatMap(Option.scala:271)
	at org.apache.spark.sql.DataFrameReader.load(DataFrameReader.scala:208)
	at org.apache.spark.s

## Stop session and clear cash from spark

In [8]:
spark.stop()
spark.catalog.clearCache()